In [ ]:
import numpy as np
import npu_driver.nn as nn

class AdvancedTargetNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Convolutional Layer 1
        self.conv1 = nn.Conv2d(1, 8)  # kernel_size=3, padding=0 고정
        self.relu1 = nn.LeakyReLU(alpha=0.25)
        
        # Convolutional Layer 2
        self.conv2 = nn.Conv2d(8, 16)  # kernel_size=3, padding=0 고정
        self.relu2 = nn.LeakyReLU(alpha=0.25)

        # Convolutional Layer 3
        self.conv3 = nn.Conv2d(16, 64)  # kernel_size=3, padding=0 고정
        self.relu3 = nn.LeakyReLU(alpha=0.25)

        # Convolutional Layer 4
        self.conv4 = nn.Conv2d(64, 128)  # kernel_size=3, padding=0 고정
        self.relu4 = nn.LeakyReLU(alpha=0.25)

        # Max Pooling Layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected Layer
        self.fc = nn.Linear(128 * 10 * 10, 10)

    def quantize_8bit(self, x):
        temp = np.floor(x)
        output_shifted = np.right_shift(temp.astype(np.int32), 9)
        quantized = np.clip(output_shifted, -128, 127).astype(np.int8).astype(np.float32)
        return quantized

    def forward(self, x):
        """입력 x: (batch_size, channels, height, width) 또는 (channels, height, width)"""
        if len(x.shape) == 3:  # (channels, height, width)
            x = np.expand_dims(x, axis=0)  # (1, channels, height, width)
        
        batch_size = x.shape[0]
        outputs = []
        
        for i in range(batch_size):
            single_input = x[i]  # (channels, height, width)
            
            # Fmap 1 (28x28 -> 26x26)
            x_out = self.conv1.forward(single_input)
            x_out = self.quantize_8bit(x_out)
            x_out = self.relu1.forward(x_out)
            
            # Fmap 2 (26x26 -> 24x24)
            x_out = self.conv2.forward(x_out)
            x_out = self.quantize_8bit(x_out)
            x_out = self.relu2.forward(x_out)
            
            # Fmap 3 (24x24 -> 22x22)
            x_out = self.conv3.forward(x_out)
            x_out = self.quantize_8bit(x_out)
            x_out = self.relu3.forward(x_out)
            
            # Fmap 4 (22x22 -> 20x20)
            x_out = self.conv4.forward(x_out)
            x_out = self.quantize_8bit(x_out)
            x_out = self.relu4.forward(x_out)
            
            # After MaxPool (20x20 -> 10x10)
            x_out = self.pool.forward(x_out)
            x_out = x_out.flatten()
            
            # Fully Connected
            x_out = self.fc.forward(x_out)
            x_out = self.quantize_8bit(x_out)
            
            outputs.append(x_out)
        
        return np.array(outputs)

    def load_weights(self, weight_files):
        """가중치 파일 로드"""
        self.conv1.set_weights(np.load(weight_files['conv1']).astype(np.float32))
        self.conv2.set_weights(np.load(weight_files['conv2']).astype(np.float32))
        self.conv3.set_weights(np.load(weight_files['conv3']).astype(np.float32))
        self.conv4.set_weights(np.load(weight_files['conv4']).astype(np.float32))
        self.fc.set_weights(np.load(weight_files['fc']).astype(np.float32))

# 사용 예시
def main():
    # 모델 초기화
    model = AdvancedTargetNetwork()
    
    # 가중치 로드
    weight_files = {
        'conv1': 'npy_files/layer1_0_weight.npy',
        'conv2': 'npy_files/layer2_0_weight.npy',
        'conv3': 'npy_files/layer3_0_weight.npy',
        'conv4': 'npy_files/layer4_0_weight.npy',
        'fc': 'npy_files/fc1_weight.npy'
    }
    model.load_weights(weight_files)
    
    # 입력 데이터 및 레이블 로드
    input_data = np.load('npy_files/input.npy')  # Shape: (10000, 1, 28, 28)
    labels = np.load('npy_files/label.npy')      # Shape: (10000,)

    # 추론 수행
    predictions = []
    for i in range(5):        # input_data.shape[0]
        input_batch = input_data[i]  # (1, 28, 28)
        output = model.forward(input_batch)  # (1, 10)
        print(output)
        predicted = np.argmax(output[0])
        predictions.append(predicted)
    
    # 정확도 계산
    correct = np.sum(np.array(predictions) == labels[0:5])
    total = len(labels[0:5])
    accuracy = (correct / total) * 100
    print(f"총 {total}개의 이미지에 대해 추론을 완료했습니다.")
    print(f"정답률: {accuracy:.2f}%")

In [2]:
main()

[[ -53.  -45.   -8.   53. -102.  -51. -128.  127.   14.   14.]]
[[  39.   28.  127.  -24.  -35.  -77.   28. -128.   11.  -52.]]
[[ -8.  75.   6. -40.  25. -10. -24. -33.  22. -31.]]
[[127. -94.  16. -30. -73. -26.  -2. -47. -12.   5.]]
[[-47. -50. -17. -69. 127. -13. -78. -12.  -2.  80.]]
총 5개의 이미지에 대해 추론을 완료했습니다.
정답률: 100.00%


In [3]:
answer = np.load('npy_files/output.npy')
print("정답 형태:", answer[0].shape)
for i in range(5):
    print("정답 레이블:", answer[i])

정답 형태: (10,)
정답 레이블: [ -53.  -45.   -8.   53. -102.  -51. -128.  127.   14.   14.]
정답 레이블: [  39.   28.  127.  -24.  -35.  -77.   28. -128.   11.  -52.]
정답 레이블: [ -8.  75.   6. -40.  25. -10. -24. -33.  22. -31.]
정답 레이블: [127. -94.  16. -30. -72. -26.  -2. -47. -12.   5.]
정답 레이블: [-47. -50. -17. -69. 127. -13. -78. -12.  -2.  80.]
